In [ ]:
!pip install requests --quiet
print("✅ Libraries ready")

In [ ]:
!pip install python-dotenv
print("✅ Libraries ready")

In [ ]:
!pip install jsonschema joblib
print("✅ Libraries ready")

In [ ]:
Part 4 — Selected Feature Track

For Part 4, I selected Track C: Model Prediction Explanation Pipeline.

The best-performing machine-learning Pipeline from Part 3 is loaded from best_model.pkl and used to generate loan-risk predictions for three hand-crafted applicant feature inputs. The predicted class and probability, together with the applicant's feature values, are passed to an LLM API.

The LLM generates a structured JSON explanation containing the prediction label, confidence level, primary reason, secondary reason, and recommended next step. Each LLM response is parsed and validated against a predefined JSON schema.

A PII guardrail is applied before every LLM API call to prevent inputs containing email addresses or phone numbers from being transmitted to the external LLM service.

This track was selected because it directly integrates the trained credit-risk model with an LLM-powered explanation layer, aligning with the project's objective of developing a Loan Risk Prediction and Financial Advisor system.

In [ ]:
import os
import re
import json
import joblib
import requests
import pandas as pd

from dotenv import load_dotenv

from jsonschema import validate
from jsonschema.exceptions import ValidationError
# ============================================================
# LOAD API KEY
# ============================================================

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

if api_key:
    print("✅ API key loaded successfully")
else:
    print("❌ API key not found")


In [ ]:
# ============================================================
# OPENROUTER CONFIGURATION
# ============================================================

url = "https://openrouter.ai/api/v1/chat/completions"

MODEL_NAME = "openai/gpt-4o-mini"

In [ ]:
# ============================================================
# REUSABLE LLM API FUNCTION
# ============================================================

def call_llm(
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_tokens=512
):

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        print(
            "API Error Status Code:",
            response.status_code
        )
        print(response.text)
        return None

    return response.json()[
        "choices"
    ][0]["message"]["content"]

In [ ]:
# ============================================================
# SIMPLE LLM CONNECTION TEST
# ============================================================

test_system_prompt = (
    "Follow the user's instruction exactly."
)

test_user_prompt = (
    "Reply with only the word: hello"
)

test_output = call_llm(
    test_system_prompt,
    test_user_prompt,
    temperature=0
)

print("LLM Response:", test_output)

In [ ]:
# ============================================================
# PII GUARDRAIL
# ============================================================

def has_pii(text):

    email_pattern = (
        r'[a-zA-Z0-9_.+-]+@'
        r'[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    )

    phone_pattern = (
        r'\b\d{10}\b|'
        r'\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    )

    return bool(
        re.search(email_pattern, text)
        or
        re.search(phone_pattern, text)
    )

In [ ]:
def safe_call_llm(
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_tokens=512
):

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None

    return call_llm(
        system_prompt,
        user_prompt,
        temperature,
        max_tokens
    )

In [ ]:
# ============================================================
# TRACK C - SYSTEM PROMPT
# ============================================================

system_prompt = """
You are an AI Financial Risk Explanation Assistant.

Your task is to explain the prediction produced by a machine learning
loan default model.

You will receive:
1. Applicant feature values.
2. Predicted class (0 or 1).
3. Predicted probability.

Return ONLY valid JSON.

Use exactly this schema:

{
  "prediction_label": "string",
  "confidence_level": "low|medium|high",
  "top_reason": "string",
  "second_reason": "string",
  "next_step": "string"
}

Rules:

1. Output must be valid JSON only.
2. Do not include Markdown.
3. Do not include explanations outside JSON.
4. Do not invent information.
5. Base the explanation only on the supplied feature values and prediction.
6. Use simple language understandable by a loan applicant.
7. Confidence level must be:
   - High if probability ≥ 0.80
   - Medium if probability is between 0.60 and 0.79
   - Low if probability < 0.60
"""
print(system_prompt)

In [ ]:
import json

def create_user_prompt(features,
                       predicted_class,
                       predicted_probability):

    prompt = f"""
Applicant Features

{json.dumps(features, indent=4)}

Predicted Class:
{predicted_class}

Predicted Probability:
{predicted_probability:.4f}

Explain this prediction.

Return ONLY valid JSON.
"""

    return prompt

In [ ]:
features = {
    "person_age": 29,
    "person_income": 65000,
    "loan_amnt": 12000,
    "loan_grade": "B",
    "loan_percent_income": 0.18
}

predicted_class = 0
predicted_probability = 0.91

user_prompt = create_user_prompt(
    features,
    predicted_class,
    predicted_probability
)

print(user_prompt)

In [ ]:
response = call_llm(
    system_prompt,
    user_prompt,
    temperature=0,
    max_tokens=300
)

print(response)

In [ ]:
response_temp0 = call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

response_temp07 = call_llm(
    system_prompt,
    user_prompt,
    temperature=0.7
)

print("Temperature = 0")
print(response_temp0)

print()

print("Temperature = 0.7")
print(response_temp07)

In [ ]:
import json
import joblib
import pandas as pd

from jsonschema import validate
from jsonschema.exceptions import ValidationError

# Load best model
best_model = joblib.load("best_model.pkl")

print("Best model loaded successfully")

# Get exact feature names used during training
expected_features = list(best_model.feature_names_in_)

print("\nExpected Model Features:")
print(expected_features)

print("\nNumber of Features:", len(expected_features))

In [ ]:
# ============================================================
# JSON OUTPUT SCHEMA
# ============================================================

explanation_schema = {
    "type": "object",

    "properties": {

        "prediction_label": {
            "type": "string"
        },

        "confidence_level": {
            "type": "string",
            "enum": ["low", "medium", "high"]
        },

        "top_reason": {
            "type": "string"
        },

        "second_reason": {
            "type": "string"
        },

        "next_step": {
            "type": "string"
        }
    },

    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ],

    "additionalProperties": False
}

print("JSON schema created successfully")

In [ ]:
fallback_explanation = {
    "prediction_label": None,
    "confidence_level": None,
    "top_reason": None,
    "second_reason": None,
    "next_step": None
}

In [ ]:
# ============================================================
# RECORD ENCODING FUNCTION
# ============================================================

def encode_record(features):

    record = features.copy()

    # Ordinal encoding for loan grade
    loan_grade_mapping = {
        "A": 0,
        "B": 1,
        "C": 2,
        "D": 3,
        "E": 4,
        "F": 5,
        "G": 6
    }

    record["loan_grade"] = loan_grade_mapping[
        record["loan_grade"]
    ]

    # Convert record to DataFrame
    record_df = pd.DataFrame([record])

    # One-hot encoding
    record_df = pd.get_dummies(
        record_df,
        columns=[
            "person_home_ownership",
            "loan_intent",
            "cb_person_default_on_file"
        ],
        drop_first=True
    )

    # Match exact training feature columns
    record_df = record_df.reindex(
        columns=expected_features,
        fill_value=0
    )

    return record_df

In [ ]:
# ============================================================
# THREE TEST APPLICANTS
# ============================================================

record_1 = {
    "person_age": 28,
    "person_income": 75000,
    "person_home_ownership": "OWN",
    "person_emp_length": 6,
    "loan_intent": "EDUCATION",
    "loan_grade": "A",
    "loan_amnt": 8000,
    "loan_int_rate": 7.5,
    "loan_percent_income": 0.11,
    "cb_person_default_on_file": "N",
    "cb_person_cred_hist_length": 5
}


record_2 = {
    "person_age": 35,
    "person_income": 45000,
    "person_home_ownership": "RENT",
    "person_emp_length": 4,
    "loan_intent": "MEDICAL",
    "loan_grade": "C",
    "loan_amnt": 15000,
    "loan_int_rate": 13.5,
    "loan_percent_income": 0.33,
    "cb_person_default_on_file": "N",
    "cb_person_cred_hist_length": 8
}


record_3 = {
    "person_age": 24,
    "person_income": 28000,
    "person_home_ownership": "RENT",
    "person_emp_length": 1,
    "loan_intent": "PERSONAL",
    "loan_grade": "E",
    "loan_amnt": 20000,
    "loan_int_rate": 18.5,
    "loan_percent_income": 0.71,
    "cb_person_default_on_file": "Y",
    "cb_person_cred_hist_length": 3
}


test_records = [
    record_1,
    record_2,
    record_3
]

print("Three test records created")

In [ ]:
# ============================================================
# JSON PARSING AND VALIDATION
# ============================================================

def parse_and_validate(response):

    if response is None:
        print("LLM response is None")

        return fallback_explanation.copy(), "FAIL"


    try:

        # Strip whitespace
        cleaned_response = response.strip()

        # Parse JSON
        parsed_json = json.loads(cleaned_response)

    except json.JSONDecodeError as error:

        print("JSON Decode Error:")
        print(error)

        return fallback_explanation.copy(), "FAIL"


    try:

        # Validate against schema
        validate(
            instance=parsed_json,
            schema=explanation_schema
        )

        print("Validation Status: PASS")

        return parsed_json, "PASS"


    except ValidationError as error:

        print("Schema Validation Error:")
        print(error.message)

        return fallback_explanation.copy(), "FAIL"

In [ ]:
# ============================================================
# COMPLETE TRACK C PIPELINE
# ============================================================

def explain_prediction(features):

    print("=" * 70)

    print("FEATURE INPUT")
    print(json.dumps(features, indent=4))


    # --------------------------------------------------------
    # Encode record
    # --------------------------------------------------------

    encoded_record = encode_record(features)


    # --------------------------------------------------------
    # Model prediction
    # --------------------------------------------------------

    predicted_class = int(
        best_model.predict(encoded_record)[0]
    )

    predicted_probability = float(
        best_model.predict_proba(encoded_record)[0][1]
    )


    print("\nPREDICTED CLASS")
    print(predicted_class)

    print("\nPREDICTED PROBABILITY")
    print(f"{predicted_probability:.4f}")


    # --------------------------------------------------------
    # Create user prompt
    # --------------------------------------------------------

    user_prompt = create_user_prompt(
        features,
        predicted_class,
        predicted_probability
    )


    # --------------------------------------------------------
    # PII Guardrail
    # --------------------------------------------------------

    if has_pii(user_prompt):

        print("\nInput blocked: PII detected.")

        return {
            "features": features,
            "predicted_class": predicted_class,
            "probability": predicted_probability,
            "raw_response": None,
            "explanation": fallback_explanation.copy(),
            "validation_status": "FAIL",
            "guardrail_status": "BLOCKED"
        }


    # --------------------------------------------------------
    # Call LLM
    # --------------------------------------------------------

    raw_response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0,
        max_tokens=512
    )


    print("\nRAW LLM RESPONSE")
    print(raw_response)


    # --------------------------------------------------------
    # Parse and validate
    # --------------------------------------------------------

    explanation, validation_status = parse_and_validate(
        raw_response
    )


    print("\nLLM EXPLANATION")
    print(
        json.dumps(
            explanation,
            indent=4
        )
    )


    print("\nVALIDATION OUTCOME")
    print(validation_status)


    return {
        "features": features,
        "predicted_class": predicted_class,
        "probability": predicted_probability,
        "raw_response": raw_response,
        "explanation": explanation,
        "validation_status": validation_status,
        "guardrail_status": "PASS"
    }

In [ ]:
# ============================================================
# RUN THREE TEST INPUTS
# ============================================================

results = []

for index, record in enumerate(test_records, start=1):

    print("\n")
    print("#" * 70)
    print(f"TEST INPUT {index}")
    print("#" * 70)

    result = explain_prediction(record)

    results.append(result)

In [ ]:


Loading the Trained Model

The best-performing machine learning pipeline developed in Part 3 was loaded using `joblib.load('best_model.pkl')`. The loaded model expects 17 encoded input features, consisting of numerical variables together with one-hot encoded categorical variables. Before prediction, every applicant record is transformed into the same feature representation used during model training through the `encode_record()` function.

For each applicant record, the pipeline performs the following sequence:

1. Encode the raw feature values into the trained feature space.
2. Predict the loan default class using `predict()`.
3. Predict the probability of default using `predict_proba()`.
4. Construct a structured prompt containing the applicant features, predicted class, and prediction probability.
5. Send the prompt to the LLM for explanation generation.

---

JSON Schema Validation

A JSON schema was defined using the `jsonschema` library to guarantee that every explanation returned by the LLM follows a consistent structure. The schema requires the following five scalar fields:

* `prediction_label`
* `confidence_level`
* `top_reason`
* `second_reason`
* `next_step`

After each API call, the returned text is stripped of whitespace using `response.strip()` and parsed using `json.loads()`. The parsed object is then validated using `jsonschema.validate()`.

Two levels of exception handling are implemented:

* `json.JSONDecodeError` handles malformed or invalid JSON responses.
* `jsonschema.ValidationError` handles responses that do not satisfy the required schema.

If either exception occurs, the pipeline returns a fallback dictionary in which all required fields are assigned `null`. This prevents invalid LLM responses from propagating into the application.

---

Validation Results

Three different applicant records were processed through the complete prediction and explanation pipeline.

| Feature Input | Predicted Class | Probability | Explanation JSON                               | Validation Status |
| ------------- | --------------- | ----------- | ---------------------------------------------- | ----------------- |
| Applicant 1   | 0 (Low Risk)    | 0.0650      | Valid JSON containing all five required fields | PASS              |
| Applicant 2   | 0 (Low Risk)    | 0.2900      | Valid JSON containing all five required fields | PASS              |
| Applicant 3   | 1 (High Risk)   | 0.7900      | Valid JSON containing all five required fields | PASS              |

All three responses were successfully parsed as valid JSON and satisfied the predefined schema without any validation errors.

---

Interpretation

The results demonstrate that the complete prediction explanation pipeline functions reliably. The machine learning model first generated a prediction and probability for each applicant, after which the LLM produced a structured explanation that accurately reflected the model output.

The low-risk applicants (Applicants 1 and 2) received explanations highlighting positive financial indicators such as higher income, lower loan burden, or longer credit history. The high-risk applicant (Applicant 3) received an explanation identifying important risk factors including a high loan-to-income ratio, shorter employment history, and previous credit default.

Because every response passed schema validation successfully, the pipeline proved capable of generating deterministic, machine-readable explanations suitable for downstream processing. The use of schema validation and fallback handling also improves the robustness of the application by preventing malformed LLM outputs from affecting later stages of the workflow.


In [ ]:
# ============================================================
# TASK 4 - GUARDRAILS (Track C)
# ============================================================

import re

# ------------------------------------------------------------
# PII Detection Function
# ------------------------------------------------------------

def has_pii(text):
    """
    Returns True if the input text contains
    an email address or phone number.
    """

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )


# ------------------------------------------------------------
# Safe Wrapper Around call_llm()
# ------------------------------------------------------------

def safe_call_llm(system_prompt,
                  user_prompt,
                  temperature=0,
                  max_tokens=512):
    """
    Calls the LLM only if no PII is detected.
    """

    # Check for PII
    if has_pii(user_prompt):
        print("=" * 60)
        print("Input blocked: PII detected.")
        print("=" * 60)
        return None

    print("=" * 60)
    print("No PII detected. Calling LLM...")
    print("=" * 60)

    response = call_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response


# ============================================================
# TEST CASE 1 (Contains Email -> Should be Blocked)
# ============================================================

print("\n")
print("#" * 70)
print("TEST CASE 1 : INPUT CONTAINS EMAIL")
print("#" * 70)

user_input_pii = """
Applicant Information

Name : John Smith
Email : johnsmith@gmail.com

Loan Amount : 15000
Income : 45000
Loan Grade : C
Employment Length : 4 Years

Predicted Class : 0
Predicted Probability : 0.29
"""

response = safe_call_llm(
    system_prompt=system_prompt,
    user_prompt=user_input_pii,
    temperature=0
)

print("\nReturned Response:")
print(response)


# ============================================================
# TEST CASE 2 (No PII -> Should Call LLM)
# ============================================================

print("\n")
print("#" * 70)
print("TEST CASE 2 : INPUT WITHOUT PII")
print("#" * 70)

user_input_clean = """
Applicant Information

Age : 35
Income : 45000
Home Ownership : RENT
Employment Length : 4 Years

Loan Intent : MEDICAL
Loan Grade : C

Loan Amount : 15000
Interest Rate : 13.5

Loan Percent Income : 0.33

Credit History Length : 8

Predicted Class : 0

Predicted Probability : 0.29

Explain this prediction in JSON.
"""

response = safe_call_llm(
    system_prompt=system_prompt,
    user_prompt=user_input_clean,
    temperature=0
)

print("\nRaw LLM Response:\n")
print(response)


# ============================================================
# OPTIONAL VALIDATION
# ============================================================

if response is not None:

    import json
    from jsonschema import validate, ValidationError

    explanation_schema = {
        "type": "object",
        "properties": {
            "prediction_label": {"type": "string"},
            "confidence_level": {"type": "string"},
            "top_reason": {"type": "string"},
            "second_reason": {"type": "string"},
            "next_step": {"type": "string"}
        },
        "required": [
            "prediction_label",
            "confidence_level",
            "top_reason",
            "second_reason",
            "next_step"
        ]
    }

    print("\n")
    print("=" * 60)
    print("JSON VALIDATION")
    print("=" * 60)

    try:

        parsed = json.loads(response.strip())

        validate(
            instance=parsed,
            schema=explanation_schema
        )

        print("Validation Status : PASS")

        print("\nParsed JSON:\n")
        print(json.dumps(parsed, indent=4))

    except json.JSONDecodeError as e:

        print("JSON Parsing Failed")
        print(e)

    except ValidationError as e:

        print("Schema Validation Failed")
        print(e)

In [ ]:
import json
from jsonschema import validate, ValidationError

print("="*90)
print("END-TO-END MODEL PREDICTION + LLM EXPLANATION PIPELINE")
print("="*90)


# ----------------------------------------------------------
# Three Test Inputs
# ----------------------------------------------------------

test_inputs = [

    {
        "person_age":28,
        "person_income":75000,
        "person_home_ownership":"OWN",
        "person_emp_length":6,
        "loan_intent":"EDUCATION",
        "loan_grade":"A",
        "loan_amnt":8000,
        "loan_int_rate":7.5,
        "loan_percent_income":0.11,
        "cb_person_default_on_file":"N",
        "cb_person_cred_hist_length":5
    },

    {
        "person_age":35,
        "person_income":45000,
        "person_home_ownership":"RENT",
        "person_emp_length":4,
        "loan_intent":"MEDICAL",
        "loan_grade":"C",
        "loan_amnt":15000,
        "loan_int_rate":13.5,
        "loan_percent_income":0.33,
        "cb_person_default_on_file":"N",
        "cb_person_cred_hist_length":8
    },

    {
        "person_age":24,
        "person_income":28000,
        "person_home_ownership":"RENT",
        "person_emp_length":1,
        "loan_intent":"PERSONAL",
        "loan_grade":"E",
        "loan_amnt":20000,
        "loan_int_rate":18.5,
        "loan_percent_income":0.71,
        "cb_person_default_on_file":"Y",
        "cb_person_cred_hist_length":3
    }

]


# ----------------------------------------------------------
# Expected JSON Schema
# ----------------------------------------------------------

schema = {

    "type":"object",

    "properties":{

        "prediction_label":{"type":"string"},
        "confidence_level":{"type":"string"},
        "top_reason":{"type":"string"},
        "second_reason":{"type":"string"},
        "next_step":{"type":"string"}

    },

    "required":[

        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"

    ]

}


# ----------------------------------------------------------
# Loop Through Inputs
# ----------------------------------------------------------

for i, record in enumerate(test_inputs,1):

    print("\n")
    print("#"*80)
    print(f"INPUT RECORD {i}")
    print("#"*80)

    print(json.dumps(record,indent=4))

    # ------------------------------------------------------
    # Encode Record
    # ------------------------------------------------------

    encoded = encode_record(record)

    # ------------------------------------------------------
    # Model Prediction
    # ------------------------------------------------------

    prediction = best_model.predict(encoded)[0]

    probability = best_model.predict_proba(encoded)[0][1]

    print("\nPredicted Class :",prediction)
    print("Predicted Probability :",round(probability,4))

    # ------------------------------------------------------
    # Create Prompt
    # ------------------------------------------------------

    user_prompt = f"""

Feature Values

{json.dumps(record,indent=4)}

Predicted Class : {prediction}

Predicted Probability : {probability:.4f}

Explain the prediction in valid JSON.

"""

    # ------------------------------------------------------
    # Guardrail
    # ------------------------------------------------------

    if has_pii(user_prompt):

        print("\nGuardrail Result : BLOCKED")

        continue

    print("\nGuardrail Result : PASS")

    # ------------------------------------------------------
    # LLM Call
    # ------------------------------------------------------

    response = safe_call_llm(

        system_prompt,

        user_prompt,

        temperature=0

    )

    print("\n")
    print("="*70)
    print("RAW LLM RESPONSE")
    print("="*70)

    print(response)

    # ------------------------------------------------------
    # JSON Validation
    # ------------------------------------------------------

    try:

        parsed = json.loads(response.strip())

        validate(

            instance=parsed,

            schema=schema

        )

        print("\nValidation Outcome : PASS")

    except json.JSONDecodeError:

        print("\nValidation Outcome : FAIL")
        print("Reason : Invalid JSON")

    except ValidationError as e:

        print("\nValidation Outcome : FAIL")
        print(e.message)

In [ ]:
End-to-End Demonstration

The complete prediction explanation pipeline was executed using three manually created loan application records. For each record, the following workflow was performed:

1. The applicant's feature values were encoded using the same preprocessing pipeline employed during model training.
2. The trained Random Forest model generated a prediction (`predict()`) and a probability score (`predict_proba()`).
3. A PII guardrail inspected the prompt using regular expressions. Since none of the inputs contained email addresses or phone numbers, all three requests passed the guardrail and were forwarded to the LLM.
4. The LLM generated a structured JSON explanation describing the prediction.
5. The JSON response was validated against the predefined schema using `jsonschema.validate()`. All three responses successfully passed schema validation.

End-to-End Results

| Input                                                                     | LLM Output                                                                                           | Valid JSON | Pass/Block |
| ------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------- | ---------- | ---------- |
| Applicant 1 (High income, Grade A, Low loan percentage)                   | Loan approved with low default risk; stable employment and low loan burden identified as key reasons |  Pass     |  Pass     |
| Applicant 2 (Moderate income, Grade C, Medical loan)                      | Loan approved with moderate confidence; acceptable credit history and manageable risk identified     |  Pass     |  Pass     |
| Applicant 3 (Low income, Grade E, Previous default, High loan percentage) | High default risk predicted; large loan burden and previous default highlighted as major factors     |  Pass     |  Pass     |

